In [26]:
pip install bs4

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ---------- ----------------------------- 1/4 [soupsieve]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   ---------------------------------------- 4/4 [bs4]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests


def test_gs_retrieval_v2():
    base_url = "https://hdpc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"

    # In Oracle HCM, parameters following the 'finder' must be semicolon/comma delimited
    # within the finder string itself for certain 'context' types.
    params = {
        "onlyData": "true",
        "expand": "requisitionList.workLocation",
        "finder": "findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC",
        "limit": 25,
        "offset": 10,
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
        "Accept": "application/json",
        "Referer": "https://houston.gs.com/",  # Some Oracle instances check the referer
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)

        # If it still fails, the API might want the limit inside the finder too
        if response.status_code == 400:
            print("Standard limit failed, trying fallback finder syntax...")
            params["finder"] = (
                "findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC,limit=5"
            )
            del params["limit"]
            response = requests.get(base_url, params=params, headers=headers)

        response.raise_for_status()
        data = response.json()

        jobs = data["items"][0]["requisitionList"]
        print(f"Successfully retrieved {len(jobs)} jobs.")
        for job in jobs:
            print(f"- {job['Title']} ({job['Id']})")

    except Exception as e:
        print(f"Retrieval failed: {e}")
        if "response" in locals():
            print(f"Response Text: {response.text}")


if __name__ == "__main__":
    test_gs_retrieval_v2()

Successfully retrieved 25 jobs.
- Core Data Platform - Software Engineering-Associate-Bengaluru (145662)
- Asset & Wealth Management, Content Marketing Manager, Associate, Dublin (160624)
- Asset & Wealth Management, Social Media, Analyst, Dublin (161799)
- Global Banking & Markets – FICC (Fixed Income, Currencies & Commodities) Engineer - Vice President - London (162477)
- Asset & Wealth Management, Operations MI Analyst, Associate, Dublin (165692)
- Asset & Wealth Management, Business Controls, Associate, Dublin (165691)
- Asset & Wealth Management, Product Manager, Analyst, Dublin (167783)
- Goldman Sachs Asset & Wealth Management - Third Party Wealth Sales - Vice President - Frankfurt (167863)
- Asset & Wealth Management- Client Solutions Group -Data Steward, Analyst/Associate -Bengaluru (168178)
- Asset & Wealth Management- CSG, Sales Enablement, Product Manager- Associate, London (168177)
- Asset & Wealth Management, Alternatives Capital Formation, Commercial Strategy, Associate,

In [ ]:
import requests
import time


def scrape_all_gs_jobs():
    base_url = "https://hdpc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions"
    limit = 25  # Keep this at 25 or 50 for stability
    offset = 0
    all_jobs = []

    while True:
        params = {
            "onlyData": "true",
            "expand": "requisitionList.workLocation",
            "finder": f"findReqs;siteNumber=CX_3002,sortBy=POSTING_DATES_DESC",
            "limit": limit,
            "offset": offset,
        }

        print(f"Fetching jobs {offset} to {offset + limit}...")
        response = requests.get(base_url, params=params)
        data = response.json()

        # Oracle HCM structure: items[0] contains the data
        root = data["items"][0]
        jobs_in_page = root.get("requisitionList", [])

        if not jobs_in_page:
            break

        all_jobs.extend(jobs_in_page)

        # Check if we've reached the end
        total_available = root.get("TotalJobsCount", 0)
        if len(all_jobs) >= total_available:
            break

        # Move to next page
        offset += limit
        time.sleep(1)  # Be nice to their server!

    print(f"Done! Scraped {len(all_jobs)} total jobs.")
    return all_jobs


# Run it
scrape_all_gs_jobs()

Fetching jobs 0 to 25...
Fetching jobs 25 to 50...
Fetching jobs 50 to 75...
Fetching jobs 75 to 100...
Fetching jobs 100 to 125...
Fetching jobs 125 to 150...
Fetching jobs 150 to 175...
Fetching jobs 175 to 200...
Fetching jobs 200 to 225...
Fetching jobs 225 to 250...
Fetching jobs 250 to 275...
Fetching jobs 275 to 300...
Fetching jobs 300 to 325...
Fetching jobs 325 to 350...
Fetching jobs 350 to 375...
Fetching jobs 375 to 400...
Fetching jobs 400 to 425...
Fetching jobs 425 to 450...
Fetching jobs 450 to 475...
Fetching jobs 475 to 500...
Fetching jobs 500 to 525...
Fetching jobs 525 to 550...
Fetching jobs 550 to 575...
Fetching jobs 575 to 600...
Fetching jobs 600 to 625...
Fetching jobs 625 to 650...
Fetching jobs 650 to 675...
Fetching jobs 675 to 700...
Fetching jobs 700 to 725...
Fetching jobs 725 to 750...
Fetching jobs 750 to 775...
Fetching jobs 775 to 800...
Fetching jobs 800 to 825...
Fetching jobs 825 to 850...
Fetching jobs 850 to 875...
Fetching jobs 875 to 900...


KeyboardInterrupt: 

In [ ]:
# HKEX

import requests
import time

def scrape_hkex_fixed_loop():
    session = requests.Session()
    base_url = "https://hkex.wd3.myworkdayjobs.com/HKEXCareerPage"
    api_url = "https://hkex.wd3.myworkdayjobs.com/wday/cxs/hkex/HKEXCareerPage/jobs"
    
    session.get(base_url)
    csrf_token = session.cookies.get("CALYPSO_CSRF_TOKEN")
    
    all_jobs = []
    limit = 20
    offset = 0
    total_available = None # Start as None to signal we haven't found it yet

    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "x-calypso-csrf-token": csrf_token
    }

    # We use a True loop and break manually based on our locked-in total
    while True:
        payload = {"appliedFacets": {}, "limit": limit, "offset": offset, "searchText": ""}
        
        try:
            response = session.post(api_url, json=payload, headers=headers)
            response.raise_for_status()
            data = response.json()

            # --- THE FIX: Only set total_available once ---
            if total_available is None:
                total_available = data.get("total", 0)
                print(f"🎯 Target Acquired: {total_available} jobs found.")

            batch = data.get("jobPostings", [])
            if not batch:
                print("📭 No more jobs returned in this batch.")
                break

            all_jobs.extend(batch)
            print(f"✅ Progress: {len(all_jobs)} / {total_available} (Offset {offset})")

            # --- BREAK CONDITION ---
            if len(all_jobs) >= total_available:
                break

            offset += limit
            time.sleep(1.5)

        except Exception as e:
            print(f"❌ Error: {e}")
            break

    print(f"\n✨ Final Count: {len(all_jobs)} jobs.")
    return all_jobs

# Run the full scrape
scrape_hkex_fixed_loop()

🎯 Target Acquired: 177 jobs found.
✅ Progress: 20 / 177 (Offset 0)
✅ Progress: 40 / 177 (Offset 20)
✅ Progress: 60 / 177 (Offset 40)
✅ Progress: 80 / 177 (Offset 60)
✅ Progress: 100 / 177 (Offset 80)
✅ Progress: 120 / 177 (Offset 100)
✅ Progress: 140 / 177 (Offset 120)
✅ Progress: 160 / 177 (Offset 140)
✅ Progress: 177 / 177 (Offset 160)

✨ Final Count: 177 jobs.


[{'title': 'Sr. Test Engineer - Associate - LMEC Testing - IT',
  'externalPath': '/job/CN-Shenzhen-HyQ/Sr-Test-Engineer---Associate---LMEC-Testing---IT_R003520',
  'locationsText': 'CN-Shenzhen-HyQ',
  'postedOn': 'Posted Today',
  'bulletFields': ['R003520']},
 {'title': 'AVP, Listing Enforcement',
  'externalPath': '/job/AVP--Listing-Enforcement_R002743',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R002743']},
 {'title': 'AVP, Listing Enforcement',
  'externalPath': '/job/HK-ONE-ES-43F/AVP--Listing-Enforcement_R003726',
  'locationsText': 'HK-ONE ES 43/F',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R003726']},
 {'title': 'Operation Officer (IPO Vetting) - 12 months contract',
  'externalPath': '/job/Operation-Officer--IPO-Vetting----12-months-contract_R003528',
  'postedOn': 'Posted Yesterday',
  'bulletFields': ['R003528']},
 {'title': 'Modern Workspace EUC Technology Lead',
  'externalPath': '/job/UK-London/Modern-Workspace-EUC-Technology-Lead_R003781',
  'loc

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_barclays_hong_kong():
    session = requests.Session()
    url = "https://search.jobs.barclays/search-jobs/results"
    base_domain = "https://search.jobs.barclays"
    
    # These are the "Magic Numbers" for Hong Kong
    params = {
        "CurrentPage": 1,
        "RecordsPerPage": 16,
        "SearchType": 3,
        "FacetTerm": "1819730",
        "FacetType": 2,
        "FacetFilters[0].ID": "1819730",
        "FacetFilters[0].FacetType": 2,
        "FacetFilters[0].Display": "Hong Kong China",
        "FacetFilters[0].IsApplied": "true",
        "OrganizationIds": "13015",
        "SearchResultsModuleName": "Search Results"
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "X-Requested-With": "XMLHttpRequest",
        "Accept": "application/json"
    }

    try:
        response = session.get(url, params=params, headers=headers)
        data = response.json()
        
        # Extract the results HTML from the JSON
        html_content = data.get("results", "")
        soup = BeautifulSoup(html_content, "html.parser")
        
        # Target the specific anchor tags with job IDs
        job_links = soup.select("a[data-job-id]")
        
        print(f"📍 Barclays Hong Kong: Found {len(job_links)} jobs on Page 1.")
        
        for link in job_links:
            title_tag = link.find(["h2", "h3"])
            title = title_tag.get_text(strip=True) if title_tag else link.get_text(strip=True)
            
            # RECONSTRUCTION: Join domain with relative path
            full_url = base_domain + link['href']
            
            print(f"- {title}")
            print(f"  Link: {full_url}")

    except Exception as e:
        print(f"Scrape failed: {e}")

all_jobs = []
current_page = 1

while True:
    params["CurrentPage"] = current_page
    response = session.get(url, params=params, headers=headers)
    data = response.json()
    
    # Extract jobs from the HTML string inside the JSON
    html_content = data.get("results", "")
    soup = BeautifulSoup(html_content, "html.parser")
    batch = soup.select("a[data-job-id]")

    # --- THE STOP CONDITION ---
    if not batch:
        print(f"🏁 No more jobs found. Total collected: {len(all_jobs)}")
        break

    all_jobs.extend(batch)
    print(f"✅ Page {current_page}: Added {len(batch)} jobs.")
    
    current_page += 1 # Move to the next page
    time.sleep(1)     # Pause to stay under the radar

📍 Barclays Hong Kong: Found 12 jobs on Page 1.
- Software Developer BA4
  Link: https://search.jobs.barclays/job/hong-kong/software-developer-ba4/13015/92372669408
- AVP Software Developer - APAC Flow Volatility Risk Pricing
  Link: https://search.jobs.barclays/job/hong-kong/avp-software-developer-apac-flow-volatility-risk-pricing/13015/91541554816
- Front Office Equity Derivatives Quant -Vice President
  Link: https://search.jobs.barclays/job/hong-kong/front-office-equity-derivatives-quant-vice-president/13015/91294068400
- Quant Prime Services (QPS) Client Service, Assistant Vice President
  Link: https://search.jobs.barclays/job/hong-kong/quant-prime-services-qps-client-service-assistant-vice-president/13015/93030514192
- Senior KDB developer
  Link: https://search.jobs.barclays/job/hong-kong/senior-kdb-developer/13015/93993356304
- Senior Application Support Engineer - AVP
  Link: https://search.jobs.barclays/job/hong-kong/senior-application-support-engineer-avp/13015/93981875520
-